# DFM Smoke Test — 0.1% HC-PT, 4 Epochs

This notebook exercises the full pipeline end-to-end:

1. Download **0.1%** of train / validation / test splits
2. Train for **4 epochs** and record all metrics (loss, teacher-forced, generative)
3. Plot metric curves
4. Run **inference** on sample spectra and compare to ground truth
5. Run **generative evaluation** on the test subset

**Runtime:** ~10–30 min on GPU (longer on CPU). Set `HF_TOKEN` for faster HuggingFace downloads.

In [ ]:
# Optional: install notebook extras (safe to re-run)
%pip install -q matplotlib pandas

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from torch.optim import AdamW

# Project root = parent of notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "src").exists():
    ROOT = ROOT.parent  # fallback if cwd is elsewhere

SCRIPTS = ROOT / "scripts"
SRC = ROOT / "src"
for path in (str(SCRIPTS), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

from config.defaults import DEFAULTS
from data.data import build_dataloader, build_vocabulary, get_dataset, get_output_aa_masses
from eval.evaluate import evaluate_generative
from eval.metrics import format_metrics
from flow_matching.scheduler import cosine_scheduler
from inference.predict import predict_peptide
from train.factory import build_models
from train.io import TrainingRunLogger, build_checkpoint_payload, load_checkpoint, load_models_from_checkpoint
from train.train import training_loop

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {ROOT}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
CFG = {
    # Data
    "subset": "[:0.1%]",          # 0.1% of each split
    "cache_dir": str(ROOT / "data" / "cache"),
    "top_k_peaks": DEFAULTS.data.top_k_peaks,
    # Training
    "epochs": 4,
    "batch_size": 16 if device.type == "cuda" else 4,
    "num_workers": 0,              # 0 avoids multiprocessing issues in notebooks
    "lr": DEFAULTS.train.learning_rate,
    "weight_decay": DEFAULTS.train.weight_decay,
    # Eval during training
    "eval_every": 1,               # generative metrics every epoch
    "eval_max_batches": 8,
    "inference_steps": 20,         # fewer steps for smoke test speed
    "noising_scheme": DEFAULTS.train.noising_scheme,
    "guidance_scale": DEFAULTS.train.guidance_scale,
    # Artifacts
    "output_dir": str(ROOT / "artifacts"),
    "run_name": "notebook_smoke_0p1pct_4ep",
}

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if HF_TOKEN:
    print("HF_TOKEN: set")
else:
    print("HF_TOKEN: not set (public download still works)")

pd.Series(CFG).to_frame("value")

## 1. Download dataset (0.1% subsets)

In [ ]:
splits = {}
for split_name in ("train", "validation", "test"):
    hf_split = f"{split_name}{CFG['subset']}"
    print(f"Loading {hf_split} ...")
    ds = get_dataset(split=hf_split, cache_dir=CFG["cache_dir"], token=HF_TOKEN)
    splits[split_name] = ds
    print(f"  rows={len(ds):,}  columns={ds.column_names}")

train_ds, valid_ds, test_ds = splits["train"], splits["validation"], splits["test"]

## 2. Build vocabulary, dataloaders, and models

In [ ]:
vocabulary = build_vocabulary(train_ds)
aa_masses = get_output_aa_masses(vocabulary).to(device)
print(f"Vocabulary size: {len(vocabulary)} tokens")
print(f"Special tokens: pad={vocabulary['<pad>']}, mask={vocabulary['<mask>']}")

train_loader = build_dataloader(
    train_ds, vocabulary,
    batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
)
valid_loader = build_dataloader(
    valid_ds, vocabulary,
    batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"],
)
test_loader = build_dataloader(
    test_ds, vocabulary,
    batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"],
)
for loader in (train_loader, valid_loader, test_loader):
    loader.dataset.top_k = CFG["top_k_peaks"]

print(f"Train batches/epoch: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches: {len(test_loader)}")

spectrum_encoder, length_predictor, decoder, guidance = build_models(vocabulary, device)
param_count = sum(p.numel() for p in [
    *spectrum_encoder.parameters(),
    *length_predictor.parameters(),
    *decoder.parameters(),
    *guidance.parameters(),
])
print(f"Total parameters: {param_count:,}")

## 3. Train for 4 epochs (record all metrics)

In [ ]:
run_logger = TrainingRunLogger(Path(CFG["output_dir"]), CFG["run_name"])
run_logger.log_run_start({"run_name": CFG["run_name"], "config": CFG, "defaults": DEFAULTS.to_dict()})

optimizer = AdamW(
    list(spectrum_encoder.parameters())
    + list(length_predictor.parameters())
    + list(decoder.parameters())
    + list(guidance.parameters()),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

args_dict = {**CFG, "device": str(device)}


def on_epoch_end(*, epoch, total_epochs, history, train_metrics, valid_metrics, weights, generative_metrics=None):
    payload = build_checkpoint_payload(
        epoch=epoch,
        total_epochs=total_epochs,
        history=history,
        vocabulary=vocabulary,
        args=args_dict,
        optimizer=optimizer,
        spectrum_encoder=spectrum_encoder,
        length_predictor=length_predictor,
        decoder=decoder,
        guidance=guidance,
        best_valid_loss=run_logger.best_valid_loss,
    )
    run_logger.log_epoch(
        epoch=epoch,
        total_epochs=total_epochs,
        history=history,
        train_metrics=train_metrics,
        valid_metrics=valid_metrics,
        weights=weights,
        checkpoint_payload=payload,
        generative_metrics=generative_metrics,
    )


history = training_loop(
    optimizer=optimizer,
    epochs=range(CFG["epochs"]),
    vocabulary=vocabulary,
    guidance=guidance,
    spectrum_encoder=spectrum_encoder,
    length_predictor=length_predictor,
    decoder=decoder,
    aa_masses=aa_masses,
    train_loader=train_loader,
    valid_loader=valid_loader,
    scheduler=cosine_scheduler,
    device=device,
    total_epochs=CFG["epochs"],
    epoch_end_callback=on_epoch_end,
    eval_every=CFG["eval_every"],
    eval_max_batches=CFG["eval_max_batches"],
    inference_steps=CFG["inference_steps"],
    noising_scheme=CFG["noising_scheme"],
    guidance_scale=CFG["guidance_scale"],
)

print("\n=== Training complete ===")
print(f"Best valid loss:      {run_logger.best_valid_loss:.4f}")
print(f"Best peptide recall:  {run_logger.best_peptide_recall:.4f}")
print(f"Latest checkpoint:    {run_logger.latest_ckpt}")
print(f"Best loss checkpoint: {run_logger.best_ckpt}")
print(f"Best recall ckpt:     {run_logger.best_metric_ckpt}")
print(f"Metrics JSONL:        {run_logger.metrics_jsonl}")

## 4. Metrics summary and plots

In [ ]:
# Per-epoch metrics from metrics.jsonl
records = []
with open(run_logger.metrics_jsonl) as f:
    for line in f:
        records.append(json.loads(line))

rows = []
for rec in records:
    row = {
        "epoch": rec["epoch_1_indexed"],
        "lambda": rec["weights"]["lambda"],
        "gamma": rec["weights"]["gamma"],
        "train_loss": rec["train"]["loss"],
        "valid_loss": rec["valid"]["loss"],
        "tf_token_acc": rec["valid"].get("token_accuracy"),
        "tf_length_acc": rec["valid"].get("length_accuracy"),
        "tf_exact_pep": rec["valid"].get("exact_peptide_accuracy"),
    }
    gen = rec.get("generative") or {}
    row.update({
        "aa_precision": gen.get("aa_precision"),
        "aa_recall": gen.get("aa_recall"),
        "peptide_precision": gen.get("peptide_precision"),
        "peptide_recall": gen.get("peptide_recall"),
        "exact_generative": gen.get("exact_peptide_accuracy"),
        "length_accuracy": gen.get("length_accuracy"),
    })
    rows.append(row)

metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

metrics_df.to_csv(run_logger.run_dir / "notebook_metrics.csv", index=False)
print(f"Saved: {run_logger.run_dir / 'notebook_metrics.csv'}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
epochs = metrics_df["epoch"]

axes[0, 0].plot(epochs, metrics_df["train_loss"], marker="o", label="train")
axes[0, 0].plot(epochs, metrics_df["valid_loss"], marker="o", label="valid")
axes[0, 0].set_title("Loss")
axes[0, 0].set_xlabel("epoch")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs, metrics_df["aa_precision"], marker="o", label="AA precision")
axes[0, 1].plot(epochs, metrics_df["aa_recall"], marker="o", label="AA recall")
axes[0, 1].set_title("Amino-acid metrics (generative)")
axes[0, 1].set_xlabel("epoch")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs, metrics_df["peptide_precision"], marker="o", label="peptide P")
axes[1, 0].plot(epochs, metrics_df["peptide_recall"], marker="o", label="peptide R")
axes[1, 0].set_title("Peptide metrics (generative)")
axes[1, 0].set_xlabel("epoch")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(epochs, metrics_df["tf_token_acc"], marker="o", label="token acc (TF)")
axes[1, 1].plot(epochs, metrics_df["tf_length_acc"], marker="o", label="length acc (TF)")
axes[1, 1].plot(epochs, metrics_df["length_accuracy"], marker="o", label="length acc (gen)")
axes[1, 1].set_title("Accuracy")
axes[1, 1].set_xlabel("epoch")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = run_logger.run_dir / "notebook_metrics.png"
plt.savefig(plot_path, dpi=120)
plt.show()
print(f"Saved: {plot_path}")

## 5. Inference — sample predictions vs ground truth

In [ ]:
# Load best checkpoint (by peptide recall if available, else best valid loss)
ckpt_path = run_logger.best_metric_ckpt if run_logger.best_metric_ckpt.exists() else run_logger.best_ckpt
checkpoint = load_checkpoint(ckpt_path, map_location=device)
load_models_from_checkpoint(
    checkpoint, spectrum_encoder, length_predictor, decoder, guidance
)
print(f"Loaded checkpoint: {ckpt_path}  (epoch {checkpoint['epoch'] + 1})")

spectrum_encoder.eval()
length_predictor.eval()
decoder.eval()
guidance.eval()

In [ ]:
from data.data import invert_vocabulary

index_to_token = invert_vocabulary(vocabulary)
pad_id = vocabulary["<pad>"]
mask_id = vocabulary["<mask>"]


def decode_ground_truth(sequence_tensor, length_tensor):
    """Convert batch tensors to peptide strings."""
    strings = []
    for row_idx in range(sequence_tensor.shape[0]):
        chars = []
        true_len = int(length_tensor[row_idx].item())
        for col_idx in range(true_len):
            token_id = int(sequence_tensor[row_idx, col_idx].item())
            if token_id not in (pad_id, mask_id):
                chars.append(index_to_token[token_id])
        strings.append("".join(chars))
    return strings


NUM_INFER_SAMPLES = 8
inference_rows = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        if batch_idx * CFG["batch_size"] >= NUM_INFER_SAMPLES:
            break

        (
            mz_array, intensity_array, precursor_mass, precursor_charge,
            sequence, mz_complementary, length, _padded_mask, spectrum_mask,
        ) = tuple(t.to(device) if torch.is_tensor(t) else t for t in batch)

        _token_ids, pred_lengths, pred_sequences = predict_peptide(
            mz_array=mz_array,
            intensity_array=intensity_array,
            precursor_mass=precursor_mass,
            precursor_charge=precursor_charge,
            mz_complementary=mz_complementary,
            spectrum_mask=spectrum_mask,
            vocabulary=vocabulary,
            spectrum_encoder=spectrum_encoder,
            length_predictor=length_predictor,
            decoder=decoder,
            guidance=guidance,
            scheduler=cosine_scheduler,
            num_steps=CFG["inference_steps"],
            noising_scheme=CFG["noising_scheme"],
            guidance_scale=CFG["guidance_scale"],
        )

        true_sequences = decode_ground_truth(sequence, length)
        for i, (pred, true) in enumerate(zip(pred_sequences, true_sequences)):
            inference_rows.append({
                "sample": batch_idx * CFG["batch_size"] + i,
                "pred_length": int(pred_lengths[i]),
                "true_length": int(length[i]),
                "predicted": pred,
                "ground_truth": true,
                "exact_match": pred == true,
            })
            if len(inference_rows) >= NUM_INFER_SAMPLES:
                break

infer_df = pd.DataFrame(inference_rows)
display(infer_df)
infer_df.to_csv(run_logger.run_dir / "notebook_inference_samples.csv", index=False)

## 6. Generative evaluation on test subset

In [ ]:
test_metrics = evaluate_generative(
    test_loader,
    vocabulary,
    spectrum_encoder,
    length_predictor,
    decoder,
    guidance,
    cosine_scheduler,
    device,
    max_batches=CFG["eval_max_batches"],
    num_steps=CFG["inference_steps"],
    noising_scheme=CFG["noising_scheme"],
    guidance_scale=CFG["guidance_scale"],
    aa_mass_tolerance=DEFAULTS.eval.aa_mass_tolerance,
    prefix_mass_tolerance=DEFAULTS.eval.prefix_mass_tolerance,
)

print(format_metrics(test_metrics))
print()
test_metrics_dict = test_metrics.to_dict()
for key, value in test_metrics_dict.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")

test_metrics_path = run_logger.run_dir / "notebook_test_metrics.json"
test_metrics_path.write_text(json.dumps(test_metrics_dict, indent=2) + "\n")
print(f"\nSaved: {test_metrics_path}")

## 7. Smoke-test checklist

If all cells ran without error, the pipeline is healthy. Expect **low** metrics after only 4 epochs on 0.1% — the goal here is correctness, not SOTA performance.

In [ ]:
checks = {
    "training completed": len(history.get("train_loss", [])) == CFG["epochs"],
    "valid loss finite": all(v == v for v in history.get("valid_loss", [])),
    "generative metrics recorded": len(history.get("valid_peptide_recall", [])) > 0,
    "checkpoint exists": run_logger.latest_ckpt.exists(),
    "inference produced sequences": len(infer_df) > 0 and infer_df["predicted"].str.len().gt(0).all(),
    "test eval ran": test_metrics.num_samples > 0,
}

check_df = pd.DataFrame({"check": list(checks.keys()), "passed": list(checks.values())})
display(check_df)

if all(checks.values()):
    print("\n✓ All smoke-test checks passed. Ready for full training on GCP.")
else:
    print("\n✗ Some checks failed — inspect logs above.")